In [ ]:
# 
# indices = [
#     (*f.split("_")[:-1], torch.load(os.path.join(indices_dir, f)))
#     for f in os.listdir(indices_dir)
#     if os.path.isfile(os.path.join(indices_dir, f)) and f.endswith((".pt"))
# ]
# indices = [(int(i.replace("iter","")),t, [int(j) for j in ind]) for i,t, ind in indices]
# df = pd.DataFrame(indices, columns=["interation", "subset", "indices"])
# df

In [ ]:
from itertools import chain
from influence_estimation.estimator import BaseEstimator
class CoresetEstimator(BaseEstimator):
    def __init__(self, full_train_dataset, model_path="./models/OLMo-2-0425-1B_tulu-v2-sft-mixture"):
        self.full_train_dataset = full_train_dataset
        self._coreset_dataset = None 
        self.train_indices = None
        self._train_dataset = None 
        def add_index(example, idx):
            example["indices"] = idx
            return example
        self.full_train_dataset = self.full_train_dataset.map(add_index, with_indices=True, num_proc=10)
            
        self.model_path = model_path
        self.train_indices_path = os.path.join(model_path, "indices")
        self.load_indices()

        self.__name__ = os.path.basename(self.model_path)
    def load_indices(self):
        indices_dir = os.path.join("./models", self.model_path, "indices")
        indices = [
        (*f.split("_")[:-1], torch.load(os.path.join(self.train_indices_path, f)))
            for f in os.listdir(indices_dir)
                if os.path.isfile(os.path.join(indices_dir, f)) and f.endswith((".pt"))
        ]
        indices = [(int(i.replace("iter","")),t, [int(j) for j in ind]) for i,t, ind in indices]
    
        
        df = pd.DataFrame(indices, columns=["iteration", "subset", "indices"]).sort_values(by="iteration")        
        self.train_indices = {
            subset: {
                iteration: group[group["subset"] == subset]["indices"].tolist()[0]
                for iteration, group in df.groupby("iteration")
            }
            for subset in df["subset"].unique()  
        }
    @property
    def coreset_dataset(self):
        if self._coreset_dataset is None:           
            self._coreset_dataset = self.full_train_dataset.select(
                set(chain.from_iterable(self.train_indices["selected"].values()))
            )
        return self._coreset_dataset
    @property
    def train_dataset(self):
        if self._train_dataset is None:
            if self.train_indices is not None:
                self._train_dataset = self.full_train_dataset.select(
                    set(chain.from_iterable(self.train_indices["full"].values()))
                )
            else:
                self._train_dataset = self.full_train_dataset
        return self._train_dataset
        